In [1]:
import langchain
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()  # loads .env file



True

In [3]:
from langchain.chat_models import init_chat_model
from langchain.messages import SystemMessage,HumanMessage

d:\GenAI\GenAI_venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [4]:
model=init_chat_model("groq:llama-3.1-8b-instant", temperature=0.8)

In [5]:
from langchain_core.prompts import ChatPromptTemplate  
from langchain_core.output_parsers import StrOutputParser

story_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a creative storyteller. Write a short, engaging story based on the given theme."),
    ("user", "Theme: {theme}\nMain character: {character}\nSetting: {setting}")
])

analysis_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a literary critic. Analyze the following story and provide insights."),
    ("user", "{story}")
])

chain = (
    story_prompt
    | model
    | StrOutputParser()
    | (lambda story: {"story": story})
    | analysis_prompt
    | model
    | StrOutputParser()
)

In [6]:
result = chain.invoke({
    "theme": "courage",
    "character": "a young girl",
    "setting": "a haunted forest"
})
print(result)

**Analyzing "The Bravery of Ember"**

**Themes:**

1. **Courage and Determination:** Ember's unwavering resolve and bravery in the face of fear and danger are the central themes of the story. Her determination to find her missing grandmother serves as a catalyst for her growth and self-discovery.
2. **Overcoming Fears:** The story highlights the importance of facing and overcoming one's fears. Ember's journey through the haunted forest serves as a metaphor for the process of confronting and overcoming the unknown.
3. **Family and Legacy:** The story emphasizes the bond between Ember and her grandmother, and the legacy that they share. The wooden box and the ancient artifacts in the chamber symbolize the passing down of knowledge and tradition.

**Symbolism:**

1. **The Haunted Forest:** The forest represents a place of uncertainty and fear, where one must face their deepest anxieties. However, it also holds the key to unlocking secrets and discovering hidden knowledge.
2. **The Wooden 

In [7]:
from langchain_core.runnables import (
    RunnablePassthrough,   # passes input unchanged, or use .assign() to add keys
    RunnableLambda,        # wraps any Python function into a runnable
    RunnableParallel,      # runs multiple chains in parallel
    RunnableSequence,  
            # chains runnables sequentially (same as using |)
)
from langchain_core.prompts import ChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [22]:
def preprocessing(data:dict)->dict:
    data["passage"]=data["passage"].strip()
    data["target_language"]=data["target_language"].strip().title()
    return data
clean_step=RunnableLambda(preprocessing)
meta_data=RunnablePassthrough.assign(
    word_count=lambda x:len(x["passage"].split())
)
translation_chain=ChatPromptTemplate.from_template(
    "Translate the following passage to {target_language}."
        "Return only the translated text, nothing else.\n\n"
        "Passage: {passage}"

)|model|StrOutputParser()
summary_chain=ChatPromptTemplate.from_template(
    "Summarize the following passage in 2-3 sentences.\n\n"
        "Passage: {passage}"
)|model|StrOutputParser()

parallel_chain=RunnablePassthrough.assign(
    translation=translation_chain,
    summary=summary_chain
)
final_prompt=ChatPromptTemplate.from_template(
   "Create a bilingual report with the following:\n\n"
    "Original Summary (English): {summary}\n\n"
    "Translated Passage: {translation}\n\n"
    "Word Count of Original Passage: {word_count}\n\n"  # ← now it's used
    "Format with clear headings for each section."
)
report_step= RunnableSequence(
    final_prompt,
    model,
    StrOutputParser()

)
pipeline=clean_step|meta_data|parallel_chain|report_step
passage = """
    Artificial intelligence is transforming industries across the globe.
    From healthcare to finance, AI systems are being deployed to automate
    tasks, improve decision-making, and unlock new capabilities that were
    previously unimaginable. Experts predict this trend will only accelerate
    in the coming decades as computing power continues to grow.
"""

result = pipeline.invoke({
    "passage": passage,
    "target_language": "hindi"   
})

print(result)


**Artificial Intelligence Revolution: A Global Trend**

**Original Summary (English)**

Artificial intelligence is revolutionizing various industries worldwide by automating tasks, enhancing decision-making, and unlocking new possibilities. This trend is expected to continue intensifying as computing power expands. Experts forecast a significant increase in AI applications across industries in the coming decades.

**Translated Passage (Hindi)**

कृत्रिम बुद्धिमत्ता पूरे विश्व में उद्योगों को बदल रही है।

स्वास्थ्य सेवा से वित्त तक, AI सिस्टम का उपयोग करके कार्यों को ऑटोमेट करने, निर्णय लेने को बेहतर बनाने, और पहले से असंभव होने वाली नए क्षमताओं को अनलॉक करने के लिए शुरू किया गया है।

विशेषज्ञों का अनुमान है कि यह चल रही प्रवृत्ति आगामी दशकों में तेज होगी, क्योंकि कंप्यूटिंग की शक्ति जारी रहेगी।

**Report Statistics**

- **Original Passage Word Count:** 47
- **Translated Passage Word Count:** 47 (Same as the original passage)

Note: The word count of the translated passage remains the s